In [ ]:
import numpy as np
from scipy.io.wavfile import read, write

INPUT = "music.wav"
OUTPUT = "soln.wav"
FLAG = "kernel{m4th15c0n5t4nt}"
CHARSET = "abcdefghijklmnopqrstuvwxyz0123456789{}"

SLOT_TIME = 0.6       
BEEP_TIME = 0.4       
HIDDEN_VOLUME = 0.15  
MUSIC_VOLUME = 0.8

def get_fibonacci(n):
    fib = [1, 1]
    for _ in range(n):
        fib.append(fib[-1] + fib[-2])
    return fib

sample_rate, audio = read(INPUT)
audio = audio.astype(float)
if audio.ndim > 1:
    audio = audio.mean(axis=1)

audio /= np.max(np.abs(audio))
audio *= MUSIC_VOLUME

samples_per_slot = int(sample_rate * SLOT_TIME)
samples_per_beep = int(sample_rate * BEEP_TIME)
hidden = np.zeros(len(audio))

total_slots = len(FLAG) + 1
audio_duration = len(audio) / sample_rate
raw_start = (audio_duration - (total_slots * SLOT_TIME)) / 2
start_offset_seconds = round(raw_start / SLOT_TIME) * SLOT_TIME
offset_samples = int(start_offset_seconds * sample_rate)

fib_key = get_fibonacci(len(FLAG))
current_freq = 15000
frequencies_played = [current_freq]

print("Generating Fibonacci + Custom Index Audio...")

for i in range(total_slots):
    slot_start = offset_samples + (i * samples_per_slot)
    beep_end = slot_start + samples_per_beep
    
    if beep_end > len(audio):
        break
        
    if i > 0:
        char = FLAG[i - 1]
        fib_val = fib_key[i - 1]
        
        char_index = CHARSET.index(char) + 1
        
        # XOR the index with the Fibonacci number
        encrypted_val = char_index ^ (fib_val % 128)
        
        # Calculate the jump (25 Hz step)
        jump = encrypted_val * 25

        if current_freq + jump <= 21500:
            current_freq = current_freq + jump
        else:
            current_freq = current_freq - jump
            
        frequencies_played.append(current_freq)
    
    t = np.arange(beep_end - slot_start) / sample_rate
    envelope = np.hanning(len(t))
    hidden[slot_start:beep_end] = HIDDEN_VOLUME * np.sin(2 * np.pi * current_freq * t) * envelope

challenge = audio + hidden
challenge_norm = challenge / np.max(np.abs(challenge))
challenge_out = np.int16(challenge_norm * 32767)
write(OUTPUT, sample_rate, challenge_out)

print(f"Success! The audio has been generated and saved as {OUTPUT}")